# Clustering for Customer Segmentation

**NOTE: The answer is at the bottom of the notebook.**

Customer Segmentation is the subdivision of a market into discrete customer groups that share similar characteristics. Customer Segmentation can be a powerful means to identify unsatisfied customer needs. Using the above data companies can then outperform the competition by developing uniquely appealing products and services.

The most common ways in which businesses segment their customer base are:
1. Demographic information, such as gender, age, familial and marital status, income, education, and occupation.
2. Geographical information, which differs depending on the scope of the company. For localized businesses, this info might pertain to specific towns or counties. For larger companies, it might mean a customer’s city, state, or even country of residence.
3. Psychographics, such as social class, lifestyle, and personality traits.
4. Behavioral data, such as spending and consumption habits, product/service usage, and desired benefits.

Advantages of Customer Segmentation
1. Determine appropriate product pricing.
2. Develop customized marketing campaigns.
3. Design an optimal distribution strategy.
4. Choose specific product features for deployment.
5. Prioritize new product development efforts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
import warnings
warnings.filterwarnings("ignore")

We are using the data from the Mall Customer Segmentation Data competition held on Kaggle.

In [ ]:
df = pd.read_csv('data/Mall_Customers.csv')

In [ ]:
df.head(5).style.hide(axis="index")

In [ ]:
df.drop(["CustomerID"], axis = 1, inplace=True)

Now we will have a short look on how the data looks like:

First we will have a glimpse on how the age of the customers is distributed

In [ ]:
df.describe().T.style \
    .format('{:.2f}')

In [ ]:
fig=plt.figure(figsize=(10,6))

ax=sns.kdeplot(df["Age"],color="black",fill=True)

fig.patch.set_facecolor('#f6f5f5')
ax.set_facecolor('#f6f5f5')

#ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
fig.text(0.3,1,"Distribution of Age", {'font': 'Serif','weight':'bold', 'size': '22','color':'black'}, alpha = 0.7)

fig.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.title("Ages Frequency")
sns.axes_style("dark")
sns.violinplot(data=df,x="Gender", y="Age")
plt.show()

Most of the customers are in their 30s. But how are the genders distributed?

In [ ]:
x = df['Gender'].value_counts()
x_male_percentage = int(x['Male']/len(df) * 100)
x_female_percentage = int(x['Female']/len(df) * 100)
print(f'Male: {x_male_percentage}%\nFemale: {x_female_percentage}%' )

In [ ]:

fig,ax=plt.subplots(figsize=(6,6))
ax.barh([1],x.values[1],height=0.7,color='black',alpha=0.7)
plt.text(-50,1, 'Male', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)
plt.text(50,1, f'{x_male_percentage}%', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)



ax.barh([0],x.values[0],height=0.7,color='#b20710',alpha=0.7)
plt.text(-50,0, 'Female', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)
plt.text(50,0, f'{x_female_percentage}%', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)


plt.text(-50,1.77, 'How Gender is Distributed? - Male vs Female' ,{'font': 'Serif', 'size': '25','weight':'bold', 'color':'black'}, alpha = 0.9)
plt.text(100,1.65, 'Male ', {'font': 'Serif','weight':'bold','size': '16', 'color':'black'}, alpha = 0.8)
plt.text(120,1.65, '|', {'color':'black' , 'size':'16', 'weight': 'bold'}, alpha = 0.9)
plt.text(130,1.65, 'Female', {'font': 'Serif','weight':'bold', 'size': '16','color':'#b20710'}, alpha = 0.7)


fig.patch.set_facecolor('#f6f5f5')
ax.set_facecolor('#f6f5f5')

ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

There are slightly more female customers than male.

Now we will look at the Annual Income and the Spending Score of the customers.

In [ ]:
fig=plt.figure(figsize=(10,6))
ax=sns.kdeplot(df["Annual Income (k$)"],color="#b20710",fill=True)
fig.patch.set_facecolor('#f6f5f5')
ax.set_facecolor('#f6f5f5')
fig.text(0.25,1,"Distribution of Annual Income (k$)", {'font': 'Serif','weight':'bold', 'size': '22','color':'black'}, alpha = 0.7)

#ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

In [ ]:
plt.figure(figsize=(15,6))
plt.subplot(1,2,1)
sns.boxplot(y=df["Spending Score (1-100)"], color="red")
plt.subplot(1,2,2)
sns.boxplot(y=df["Annual Income (k$)"])
plt.show()

In [ ]:
ss1_20 = df["Spending Score (1-100)"][(df["Spending Score (1-100)"] >= 1) & (df["Spending Score (1-100)"] <= 20)]
ss21_40 = df["Spending Score (1-100)"][(df["Spending Score (1-100)"] >= 21) & (df["Spending Score (1-100)"] <= 40)]
ss41_60 = df["Spending Score (1-100)"][(df["Spending Score (1-100)"] >= 41) & (df["Spending Score (1-100)"] <= 60)]
ss61_80 = df["Spending Score (1-100)"][(df["Spending Score (1-100)"] >= 61) & (df["Spending Score (1-100)"] <= 80)]
ss81_100 = df["Spending Score (1-100)"][(df["Spending Score (1-100)"] >= 81) & (df["Spending Score (1-100)"] <= 100)]

ssx = ["1-20", "21-40", "41-60", "61-80", "81-100"]
ssy = [len(ss1_20.values), len(ss21_40.values), len(ss41_60.values), len(ss61_80.values), len(ss81_100.values)]

plt.figure(figsize=(15,6))
sns.barplot(x=ssx, y=ssy, palette=['grey', 'grey','red','grey','grey'])
ax.set_facecolor('#f5f6f6')
plt.title("Spending Scores")
plt.xlabel("Score")
plt.ylabel("Number of Customer Having the Score")
plt.show()

In [ ]:

ai0_30 = df["Annual Income (k$)"][(df["Annual Income (k$)"] >= 0) & (df["Annual Income (k$)"] <= 30)]
ai31_60 = df["Annual Income (k$)"][(df["Annual Income (k$)"] >= 31) & (df["Annual Income (k$)"] <= 60)]
ai61_90 = df["Annual Income (k$)"][(df["Annual Income (k$)"] >= 61) & (df["Annual Income (k$)"] <= 90)]
ai91_120 = df["Annual Income (k$)"][(df["Annual Income (k$)"] >= 91) & (df["Annual Income (k$)"] <= 120)]
ai121_150 = df["Annual Income (k$)"][(df["Annual Income (k$)"] >= 121) & (df["Annual Income (k$)"] <= 150)]

aix = ["$ 0 - 30,000", "$ 30,001 - 60,000", "$ 60,001 - 90,000", "$ 90,001 - 120,000", "$ 120,001 - 150,000"]
aiy = [len(ai0_30.values), len(ai31_60.values), len(ai61_90.values), len(ai91_120.values), len(ai121_150.values)]

plt.figure(figsize=(15,6))
sns.barplot(x=aix, y=aiy, palette=['grey', 'grey','red','grey','grey'])
plt.title("Annual Incomes")
plt.xlabel("Income")
plt.ylabel("Number of Customer")
plt.show()

We can see that the most customers earn between $ 60.000 \$ $ and $ 90.000 \$ $ a year and have a spending score between 41 and 60. Let's now plot the annual income against the spending score:

In [ ]:
fig=plt.figure(figsize=(10,6))
ax=sns.scatterplot(x=df["Annual Income (k$)"],y=df["Spending Score (1-100)"],color="#b20710")

fig.patch.set_facecolor('#f6f5f5')

ax.set_facecolor('#f5f6f6')
for loc in ['right', 'top']:
    ax.spines[loc].set_visible(False)
 
fig.text(0.2,1,"Plotting Annual Income (k$) against the Spending Score",**{'font':'serif', 'size':18,'weight':'bold'}, alpha = 1)
fig.text(0.1,0.90,"It seems like there are already some clusters :",**{'font':'serif', 'size':14,}, alpha = 1)


fig.show()

It seems like that there are some groups of customers. Let's have a look at that with a clustering algorithm.

## Clustering the Customer Data

We will cluster the data by Annual Income (k$) and Spending Score. So first we will create an array that only contains this two elements.

In [ ]:
x = df.iloc[:, [2, 3]].values

Now we will use the Elbow Method to get the best number of clusters. We saw in the plot above that the data already shows 5 clusters so we will use the Elbow Method in this case to validate the number of clusters we expect.

In [ ]:
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters = i, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
    km.fit(x)
    wcss.append(km.inertia_)
fig=plt.figure(figsize=(10,6))  
fig.patch.set_facecolor('#f6f5f5')

plt.plot(range(1, 11), wcss)
plt.title('The Elbow Method', fontsize = 20)
plt.xlabel('No. of Clusters')
plt.ylabel('wcss')
fig.text(0.5,0.4,"The best k-value is 5")
plt.show()

The best number of clusters we derive from the elbow method seems to be 5, which coincides with the number of clusters we can see when we plot the Annual Income (k$) against the Spending Score. So we will choose n_clusters = 5 for our kMeans model. So lets train the model and predict the clusters.

Another method to estimate a good value for k is to have a look at the corresponding silhouette plots (also called knive plots). Let's create silhouette plots fro different numbers of k from 2 to 10. 

In [ ]:
# Range of k values
range_n_clusters = range(2, 11)

# Create a figure to hold the subplots
fig, axes = plt.subplots(5, 2, figsize=(15, 25))
axes = axes.flatten()

for k_idx, n_clusters in enumerate(range_n_clusters):
    ax = axes[k_idx]


    # Create a subplot for each k value
    ax.set_xlim([-0.1, 1])
    ax.set_ylim([0, len(x) + (n_clusters + 1) * 10])

    # Initialize the KMeans algorithm
    kmeans = KMeans(n_clusters = n_clusters, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
    cluster_labels = kmeans.fit_predict(x)

    # Compute the silhouette scores for each sample
    silhouette_avg = silhouette_score(x, cluster_labels)
    sample_silhouette_values = silhouette_samples(x, cluster_labels)

    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = plt.cm.nipy_spectral(float(i) / n_clusters)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values, facecolor=color, edgecolor=color, alpha=0.7)

        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax.set_title(f'Number of Clusters: {n_clusters}')
    ax.set_xlabel("Silhouette coefficient values")
    ax.set_ylabel("Cluster label")
    ax.axvline(x=silhouette_avg, color="red", linestyle="--")
    ax.set_yticks([])

# Adjust layout to avoid overlap
plt.tight_layout()
plt.show()

As you can derive from the plots, a number of 5 clusters might be most suitable for our data. We conclude this by haing the best average shiloutte score compared to the others and the shape of the "knives". 

In [ ]:
km = KMeans(n_clusters = 5, init = 'k-means++', max_iter = 300, n_init = 10, random_state = 0)
y_means = km.fit_predict(x)

For a quick overview how the clusters look like we can create a pandas series with the clusters and concatenate it onto the original dataset and print some samples for each cluster.

In [ ]:
df_cluster = pd.Series(y_means, name='Cluster')

In [ ]:
df_clustered = pd.concat([df, df_cluster], axis=1)

In [ ]:
for i in range(5):
    print(df_clustered[['Annual Income (k$)', 'Spending Score (1-100)', 'Cluster']].query(f'Cluster == {i}').sample(4))

And now lets plot our cluster:

In [ ]:
fig=plt.figure(figsize=(10,6))
ax=sns.scatterplot(x=df["Annual Income (k$)"],y=df["Spending Score (1-100)"], palette='viridis', hue=df_clustered['Cluster'])

for loc in ['right', 'top']:
    ax.spines[loc].set_visible(False)
 

plt.scatter(km.cluster_centers_[:,0], km.cluster_centers_[:, 1], s = 50, c = 'blue' , label = 'centroid')

plt.title('K Means Clustering', fontsize = 20)
plt.xlabel('Annual Income')
plt.ylabel('Spending Score')
plt.legend()
plt.show()

So we see that there are 5 groups of customer:

1. Low income with low spending score.
2. Low income with high spending score.
3. Average income with average spending score.
4. High income with low spending score.
5. High income with high spending score.

So of course our target group is the high income with high spending score. But we can now also think about how we get the high income with low spending score into the high spending group and conduct targeted marketing campaign. So who is in this group?

In [ ]:
df_high_income_low_spending = df_clustered[['Age','Gender', 'Annual Income (k$)', 'Spending Score (1-100)', 'Cluster']].query('Cluster == 0')

In [ ]:
# new column based on clustering criterion to split to high income low spending group
df_clustered['income_group'] = df_clustered['Cluster'].apply(lambda x: 1 if x in [1, 2, 3, 4] else 0) 
fig=plt.figure(figsize=(10,6))
ax = sns.kdeplot(data= df_clustered, x= "Age", hue="income_group", fill = True, legend = False, palette=['black', '#b20710'] )

fig.patch.set_facecolor('whitesmoke') 
ax.set_facecolor('white')

#ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
fig.text(0.3,1,"Distribution of Age", {'font': 'Serif','weight':'bold', 'size': '22','color':'black'}, alpha = 0.7)
fig.text(0.2,0.95,"Customers",**{'font':'serif', 'size':18,'weight':'bold','color':'#b20710'}, alpha = 0.6)
fig.text(0.2,0.90,"Only the high income low spending score group",**{'font':'serif', 'size':18,'weight':'bold','color':'black'}, alpha = 0.6)

plt.show()

Let's look at the gender distribution in the high income, low spending cluster.

In [ ]:
x_gender = df_high_income_low_spending['Gender'].value_counts()
x_male_percentage = int(x_gender['Male']/len(df_high_income_low_spending) * 100)
x_female_percentage = int(x_gender['Female']/len(df_high_income_low_spending) * 100)
print(f'Male: {x_male_percentage}%\nFemale: {x_female_percentage}%' )

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
ax.barh([1],x_gender.values[0],height=0.7,color='black',alpha=0.7)
plt.text(-5,1, 'Male', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)
plt.text(5,1, f'{x_male_percentage}%', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)



ax.barh([0],x_gender.values[1],height=0.7,color='#b20710',alpha=0.7)
plt.text(-5,0, 'Female', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)
plt.text(5,0, f'{x_female_percentage}%', {'font': 'Serif','weight':'bold','size': '16','style':'normal', 'color':'black'}, alpha = 0.7)


plt.text(-5,1.77, 'How Gender is Distributed in the high income, low spending cluster' ,{'font': 'Serif', 'size': '25','weight':'bold', 'color':'black'}, alpha = 0.9)
plt.text(10,1.65, 'Male ', {'font': 'Serif','weight':'bold','size': '16', 'color':'#b20710'}, alpha = 0.8)
plt.text(12,1.65, '|', {'color':'black' , 'size':'16', 'weight': 'bold'}, alpha = 0.9)
plt.text(13,1.65, 'Female', {'font': 'Serif','weight':'bold', 'size': '16','color':'black'}, alpha = 0.7)
plt.text(-5,1.5, 'Looks like most of the Customers are Male', 
        {'font':'Serif', 'size':'12.5','color': 'black'})


fig.patch.set_facecolor('#f6f5f5')
ax.set_facecolor('#f6f5f5')

ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

We can see that this group is on average older compared to the whole data set and that there are proportionally more men in this group than the whole data set.

Exercise: What happens when we include the age and the gender into the clustering, do you still see clusters? If yes how are they composed? Feel free to visualize the results using different pairwise feature combinations or even using a 3D plot.


As the plots and analysis below show, including age and gender still results in meaningful clusters, but the composition of these clusters changes. The clusters now reflect a more complex interplay between age, gender, income, and spending score.
Overall, however, the clusters are not as distinctly separated as when only income and spending score were used. This is likely due to the added complexity of the additional features, which can introduce more variability into the clustering process.

For a more detailed analysis, please refer to the code and visualizations below.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score
from mpl_toolkits.mplot3d import Axes3D
import os
import warnings
warnings.filterwarnings("ignore")

output_dir = 'clustering_output'
os.makedirs(output_dir, exist_ok=True)

# Load the data
df = pd.read_csv('data/Mall_Customers.csv')
df.drop(["CustomerID"], axis=1, inplace=True)

print("="*60)
print("CLUSTERING WITH AGE AND GENDER INCLUDED")
print("="*60)

# data prep 

# Encode Gender (Male=0, Female=1)
df['Gender_Encoded'] = df['Gender'].map({'Male': 0, 'Female': 1})

# Select features for clustering
X_all = df[['Age', 'Gender_Encoded', 'Annual Income (k$)', 'Spending Score (1-100)']]

# Standardize the features (wichtig für KMeans!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

print("\nFeatures for clustering:")
print(X_all.columns.tolist())
print(f"\nShape of data: {X_scaled.shape}")

# find optimal number of clusters

# Elbow Method
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', max_iter=300, n_init=10, random_state=0)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

# Plot Elbow and Silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method (with Age & Gender)')
ax1.grid(True, alpha=0.3)

ax2.plot(K_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Scores (with Age & Gender)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/elbow_silhouette_all_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("Silhouette Scores for different k:")
for k, score in zip(K_range, silhouette_scores):
    print(f"k={k}: {score:.4f}")
print("="*60)

# clustering with optimal k

# Let's try with 5 and 6 clusters to compare
optimal_k = 5  # You can change this based on the scores

km_all = KMeans(n_clusters=optimal_k, init='k-means++', max_iter=300, n_init=10, random_state=0)
df['Cluster_All_Features'] = km_all.fit_predict(X_scaled)

print(f"\nClustering performed with k={optimal_k}")
print("\nCluster sizes:")
print(df['Cluster_All_Features'].value_counts().sort_index())

# analysis

print("\n" + "="*60)
print("CLUSTER CHARACTERISTICS")
print("="*60)

cluster_summary = df.groupby('Cluster_All_Features').agg({
    'Age': ['mean', 'std'],
    'Gender': lambda x: (x == 'Male').sum() / len(x) * 100,  # % Male
    'Annual Income (k$)': ['mean', 'std'],
    'Spending Score (1-100)': ['mean', 'std']
}).round(2)

cluster_summary.columns = ['Age_Mean', 'Age_Std', 'Male_%', 'Income_Mean', 'Income_Std', 'Spending_Mean', 'Spending_Std']
print("\n", cluster_summary)

# visualizations

# 5.1: Pairwise Feature Plots
print("\nCreating pairwise visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(f'Pairwise Feature Plots - {optimal_k} Clusters (with Age & Gender)', fontsize=16, fontweight='bold')

# Income vs Spending
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', 
                hue='Cluster_All_Features', palette='viridis', s=100, alpha=0.7, ax=axes[0, 0])
axes[0, 0].set_title('Income vs Spending Score')

# Age vs Spending
sns.scatterplot(data=df, x='Age', y='Spending Score (1-100)', 
                hue='Cluster_All_Features', palette='viridis', s=100, alpha=0.7, ax=axes[0, 1])
axes[0, 1].set_title('Age vs Spending Score')

# Age vs Income
sns.scatterplot(data=df, x='Age', y='Annual Income (k$)', 
                hue='Cluster_All_Features', palette='viridis', s=100, alpha=0.7, ax=axes[0, 2])
axes[0, 2].set_title('Age vs Income')

# Income vs Spending with Gender as marker
for cluster in df['Cluster_All_Features'].unique():
    cluster_data = df[df['Cluster_All_Features'] == cluster]
    for gender, marker in [('Male', 'o'), ('Female', '^')]:
        gender_data = cluster_data[cluster_data['Gender'] == gender]
        axes[1, 0].scatter(gender_data['Annual Income (k$)'], 
                          gender_data['Spending Score (1-100)'],
                          label=f'Cluster {cluster} - {gender}', 
                          marker=marker, s=100, alpha=0.7)
axes[1, 0].set_xlabel('Annual Income (k$)')
axes[1, 0].set_ylabel('Spending Score (1-100)')
axes[1, 0].set_title('Income vs Spending (Gender as markers)')
axes[1, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

# Age distribution by cluster
for cluster in sorted(df['Cluster_All_Features'].unique()):
    cluster_data = df[df['Cluster_All_Features'] == cluster]
    axes[1, 1].hist(cluster_data['Age'], alpha=0.5, label=f'Cluster {cluster}', bins=15)
axes[1, 1].set_xlabel('Age')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Age Distribution by Cluster')
axes[1, 1].legend()

# Gender distribution by cluster
gender_cluster = df.groupby(['Cluster_All_Features', 'Gender']).size().unstack()
gender_cluster.plot(kind='bar', ax=axes[1, 2], color=['#1f77b4', '#ff7f0e'])
axes[1, 2].set_xlabel('Cluster')
axes[1, 2].set_ylabel('Count')
axes[1, 2].set_title('Gender Distribution by Cluster')
axes[1, 2].legend(title='Gender')
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(f'{output_dir}/pairwise_plots_all_features.png', dpi=300, bbox_inches='tight')
plt.show()

# 5.2: 3D Scatter Plot
print("\nCreating 3D visualization...")

fig = plt.figure(figsize=(14, 10))

# 3D plot 1: Income, Spending, Age
ax1 = fig.add_subplot(121, projection='3d')
scatter1 = ax1.scatter(df['Annual Income (k$)'], 
                       df['Spending Score (1-100)'], 
                       df['Age'],
                       c=df['Cluster_All_Features'], 
                       cmap='viridis', 
                       s=100, 
                       alpha=0.7,
                       edgecolors='black',
                       linewidth=0.5)
ax1.set_xlabel('Annual Income (k$)', fontsize=10)
ax1.set_ylabel('Spending Score', fontsize=10)
ax1.set_zlabel('Age', fontsize=10)
ax1.set_title('3D Clustering: Income, Spending, Age', fontsize=12, fontweight='bold')
plt.colorbar(scatter1, ax=ax1, label='Cluster', shrink=0.5)

# 3D plot 2: Same but with different angle and gender as marker
ax2 = fig.add_subplot(122, projection='3d')
for cluster in df['Cluster_All_Features'].unique():
    cluster_data = df[df['Cluster_All_Features'] == cluster]
    for gender, marker in [('Male', 'o'), ('Female', '^')]:
        gender_data = cluster_data[cluster_data['Gender'] == gender]
        ax2.scatter(gender_data['Annual Income (k$)'], 
                   gender_data['Spending Score (1-100)'], 
                   gender_data['Age'],
                   label=f'C{cluster}-{gender[0]}', 
                   marker=marker, 
                   s=100, 
                   alpha=0.7)
ax2.set_xlabel('Annual Income (k$)', fontsize=10)
ax2.set_ylabel('Spending Score', fontsize=10)
ax2.set_zlabel('Age', fontsize=10)
ax2.set_title('3D Clustering with Gender Markers', fontsize=12, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.15, 1), fontsize=8)

plt.tight_layout()
plt.savefig(f'{output_dir}/3d_clustering_all_features.png', dpi=300, bbox_inches='tight')
plt.show()

# comparison with original clustering

print("\n" + "="*60)
print("COMPARISON: Original (Income+Spending) vs Extended (All Features)")
print("="*60)

# Original clustering (only Income and Spending)
X_original = df[['Annual Income (k$)', 'Spending Score (1-100)']]
scaler_original = StandardScaler()
X_original_scaled = scaler_original.fit_transform(X_original)
km_original = KMeans(n_clusters=5, init='k-means++', max_iter=300, n_init=10, random_state=0)
df['Cluster_Original'] = km_original.fit_predict(X_original_scaled)

# Compare cluster assignments
comparison_matrix = pd.crosstab(df['Cluster_Original'], df['Cluster_All_Features'], 
                                rownames=['Original'], colnames=['Extended'])
print("\nCluster Assignment Comparison:")
print(comparison_matrix)

# Visualize comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original clustering
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', 
                hue='Cluster_Original', palette='viridis', s=100, alpha=0.7, ax=ax1)
ax1.set_title('Original Clustering\n(Income + Spending only)', fontsize=14, fontweight='bold')
ax1.scatter(km_original.cluster_centers_[:, 0] * scaler_original.scale_[0] + scaler_original.mean_[0], 
           km_original.cluster_centers_[:, 1] * scaler_original.scale_[1] + scaler_original.mean_[1], 
           s=300, c='red', marker='X', edgecolors='black', linewidth=2, label='Centroids')
ax1.legend()

# Extended clustering
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)', 
                hue='Cluster_All_Features', palette='viridis', s=100, alpha=0.7, ax=ax2)
ax2.set_title(f'Extended Clustering\n(Income + Spending + Age + Gender)', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{output_dir}/clustering_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# cluster profiling

print("\n" + "="*60)
print("DETAILED CLUSTER PROFILES")
print("="*60)

for cluster in sorted(df['Cluster_All_Features'].unique()):
    cluster_data = df[df['Cluster_All_Features'] == cluster]
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster} (n={len(cluster_data)})")
    print(f"{'='*60}")
    print(f"Age: {cluster_data['Age'].mean():.1f} ± {cluster_data['Age'].std():.1f} years")
    print(f"Gender: {(cluster_data['Gender'] == 'Male').sum()} Male ({(cluster_data['Gender'] == 'Male').sum()/len(cluster_data)*100:.1f}%), "
          f"{(cluster_data['Gender'] == 'Female').sum()} Female ({(cluster_data['Gender'] == 'Female').sum()/len(cluster_data)*100:.1f}%)")
    print(f"Income: {cluster_data['Annual Income (k$)'].mean():.1f} ± {cluster_data['Annual Income (k$)'].std():.1f} k$")
    print(f"Spending: {cluster_data['Spending Score (1-100)'].mean():.1f} ± {cluster_data['Spending Score (1-100)'].std():.1f}")
    
    # Characterization
    avg_age = cluster_data['Age'].mean()
    avg_income = cluster_data['Annual Income (k$)'].mean()
    avg_spending = cluster_data['Spending Score (1-100)'].mean()
    male_pct = (cluster_data['Gender'] == 'Male').sum() / len(cluster_data) * 100
    
    age_label = "Young" if avg_age < 35 else "Middle-aged" if avg_age < 50 else "Older"
    income_label = "Low" if avg_income < 50 else "Medium" if avg_income < 75 else "High"
    spending_label = "Low" if avg_spending < 40 else "Medium" if avg_spending < 65 else "High"
    gender_label = "Male-dominated" if male_pct > 60 else "Female-dominated" if male_pct < 40 else "Balanced"
    
    print(f"\n📊 Profile: {age_label} | {income_label} Income | {spending_label} Spending | {gender_label}")

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)
print(f"\nFiles saved in '{output_dir}/' directory:")
print(f"  - {output_dir}/elbow_silhouette_all_features.png")
print(f"  - {output_dir}/pairwise_plots_all_features.png")
print(f"  - {output_dir}/3d_clustering_all_features.png")
print(f"  - {output_dir}/clustering_comparison.png")